# 교안 01: 계획을 세우고 실행하기 (MCP 도구로)

앞 시간에 우리는 **남이 만든 MCP 서버**를 하나씩 붙여, 파일·DB·코드 실행·브라우저를 에이전트에 쥐여 줬습니다.
그런데 그때 던진 질문은 대개 **한두 단계**였습니다. 실무의 분석 요청은 다릅니다.
"조회하고, 계산하고, 그래프까지 그려서 리포트로 정리해줘". 이런 **복합 요청**을 에이전트가 우왕좌왕하지 않게 하려면,
**먼저 계획을 세우고** 단계별로 처리하게 해야 합니다.

이번 시간엔 에이전트에 **미들웨어(middleware)** 를 끼워 **할 일 목록(계획)** 을 세우게 만듭니다.
미들웨어가 루프의 어느 자리에 걸리는지(**훅**)를 먼저 보고, 계획을 세운 에이전트가 같은 복합 요청을
어떻게 다르게 처리하는지 견줍니다.

## 지난 시간 복습

- **MCP 서버**를 붙이면 우리가 만들지 않은 도구를 그대로 씁니다. 설정은 `command`·`args`·`transport` 딕셔너리 하나입니다.
- 노트북에서는 `await client.get_tools()` 로 도구를 받습니다. 도구 목록은 **서버에 물어봐야** 알 수 있어서 `await` 가 붙습니다.
- 에이전트에 **넘긴 도구만** 쓸 수 있습니다. 도구를 고르는 것이 곧 권한 설계였습니다.
- 이번 시간엔 그 도구들을 **그대로 쓰되**, 에이전트가 **계획을 세우게** 만듭니다.

## 오늘의 목표

- [ ] **복합 요청**을 계획 없이 처리할 때의 한계를 관찰한다.
- [ ] **미들웨어**가 무엇인지, 에이전트의 어디를 확장하는지 이해한다.
- [ ] **훅 여섯 자리**를 알고, 어떤 일을 **어느 자리에서** 해야 할 수 있는지 고른다.
- [ ] **`TodoListMiddleware`** 로 **Plan-and-Execute**(계획 수립 → 단계 실행)를 구현하고, 그 계획을 눈으로 확인한다.

---
## 준비

이 단원은 **MCP 서버 세 개**를 씁니다. 앞 단원에서 하나씩 다뤄 본 것들입니다.

| 서버 | 실행 | 맡는 일 | 공식 문서 |
|---|---|---|---|
| SQLite `mcp-server-sqlite` | `uvx` | 조회·집계 | https://pypi.org/project/mcp-server-sqlite/ |
| 코드 실행 `mcp-server-code-runner` | `npx` | 계산·그래프 | https://github.com/formulahendry/mcp-server-code-runner |
| 파일시스템 `@modelcontextprotocol/server-filesystem` | `npx` | 리포트 저장 | https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem |

준비물은 **Node.js**(`npx`)·**uv**(`uvx`)·**`OPENAI_API_KEY`**(일차 폴더의 `.env`)입니다.

In [ ]:
import platform
import sys
from pathlib import Path

DAY_DIR = Path.cwd().parent        # 이 노트북은 교안_01 폴더에서 연다. 그 위가 일차 폴더(day22).
sys.path.append(str(DAY_DIR))      # 일차 폴더의 utils.py 를 쓴다

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, TodoListMiddleware
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from utils import (child_env, chinook_db_path, load_api_key,
                   print_trajectory, quiet_stdio_logs, tool_names)

quiet_stdio_logs()                 # 코드 실행 서버가 stdout 에 섞어 보내는 안내문 때문에 나는 긴 경고를 끈다

DATA_DIR = DAY_DIR / "data"

load_api_key(DAY_DIR)              # 모델을 부르므로 키를 맨 앞에서 확인한다
DB_PATH = chinook_db_path(DATA_DIR)   # data 폴더의 chinook.db 경로를 돌려준다
CHILD_ENV = child_env()                 # 코드 실행 서버가 python 을 찾게 하는 환경 변수

In [ ]:
# 앞 단원에서 한 줄씩 뜯어본 설정에 한 가지를 더한다. 두 서버를 output 폴더 기준으로 띄우는 것이다.
# 산출물은 모두 일차 폴더의 output 에 모은다.
#   files_write_file : 파일 서버에 열어 준 폴더가 상대경로의 기준이 된다
#   code_run-code    : 서버 프로세스의 작업 폴더(cwd)가 상대경로의 기준이 된다
# 기준이 서로 다른 두 도구를 같은 폴더로 맞춰 두면, 모델은 파일 이름만 적으면 된다.
# 긴 경로를 프롬프트에 넣지 않는 것이 중요하다. 넣으면 모델이 그 경로를 다시 타이핑하다 오타를 내고,
# 코드 실행 서버는 그 오타를 걸러 주지 않아 엉뚱한 폴더에 조용히 저장된다.
OUTPUT_DIR = DAY_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)      # 파일 서버는 없는 폴더를 만들어 주지 않는다

SQLITE = {
    "command": "uvx",
    "args": ["--with", "mcp==1.9.4",         # 이 서버는 최신 mcp 로 띄우면 죽는다. 버전을 고정한다
             "--from", "mcp-server-sqlite",
             "mcp-server-sqlite",
             "--db-path", str(DB_PATH)],
    "transport": "stdio",
}
CODE_RUNNER = {
    "command": "npx",
    "args": ["-y", "mcp-server-code-runner"],
    "transport": "stdio",
    "env": CHILD_ENV,
    "cwd": str(OUTPUT_DIR),   # 이 서버가 돌리는 코드의 작업 폴더. savefig("그림.png") 가 여기에 떨어진다
}
FILESYSTEM = {
    "command": "npx",
    # 열어 주는 폴더가 곧 쓰기 허용 범위이자 상대경로의 기준이다
    "args": ["-y", "@modelcontextprotocol/server-filesystem", str(OUTPUT_DIR)],
    "transport": "stdio",
}


# 한글 폰트 이름은 OS 마다 다르다. "한글 폰트를 써라" 라고만 하면 모델이 없는 이름을 골라
# 제목이 네모(□□□)로 나온다. 여기서 정해 프롬프트에 넣어 준다.
FONT = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}.get(platform.system(), "NanumGothic")

In [ ]:
print("서버 세 개를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
client = MultiServerMCPClient(
    {"db": SQLITE, "code": CODE_RUNNER, "files": FILESYSTEM},
    tool_name_prefix=True,      # db_·code_·files_ 접두사를 붙인다. 서버끼리 도구 이름이 겹쳐도 충돌하지 않는다
)
mcp_tools = await client.get_tools()

# 쓰기 도구는 리포트 저장용 하나만 남긴다. 넘기지 않은 도구는 모델이 존재조차 모른다.
ALLOWED = {"db_read_query", "db_list_tables", "db_describe_table",
           "code_run-code", "files_write_file"}
TOOLS = [t for t in mcp_tools if t.name in ALLOWED]

print("에이전트에 쓸 도구:", [t.name for t in TOOLS])

> 실습 데이터는 앞 단원에서 쓴 **chinook**(가상의 음악 판매점) DB 입니다.
> `invoices`(주문)·`customers`(고객)·`employees`(직원)·`tracks`(곡) 같은 표 11개가 이어져 있습니다.

---
# 1. 계획 없는 에이전트의 한계

여러 단계가 필요한 요청을 계획 없이 던지면, 에이전트가 **어디까지 했는지 놓치거나** 일부 단계를 **건너뛰기** 쉽습니다.
사람도 복잡한 일을 할 때 **할 일 목록**을 먼저 적는 이유와 같습니다.

먼저 **계획 장치가 없는** 에이전트에 복합 요청을 던져, 어떻게 처리하는지 메시지 기록으로 봅니다.

In [ ]:
SYSTEM_BASE = (
    "너는 데이터 분석 비서다. 표의 조회·집계는 db_read_query 로 SQL 을 실행해 구하고, "
    "그 결과를 가공하는 계산과 그래프는 code_run-code 로 한다. 숫자를 암산하거나 지어내지 않는다. "
    "코드의 마지막 줄은 반드시 print 로 출력한다. 값만 적은 줄은 아무것도 돌려주지 않는다. "
    "결과가 비어 있으면 print 를 빠뜨린 것이니 print 를 넣어 다시 실행한다. "
    "경고(Stderr)만 돌아오면 이 서버가 표준 출력을 버린 것이다. 코드 맨 위에서 "
    "warnings.filterwarnings('ignore') 로 경고를 끄고 다시 실행한다. "
    "같은 코드를 두 번 보내지 않는다. "
    "code_run-code 는 한 번에 하나씩만 부른다. 이 서버는 모든 코드를 같은 임시 파일에 쓰므로 "
    "동시에 두 번 부르면 서로의 코드를 덮어써 실패한다. "
    "표나 열 이름이 확실하지 않으면 db_list_tables 와 db_describe_table 로 먼저 확인한다. "
    "그림은 files_write_file 로 만들 수 없다. 그림은 code_run-code 안에서 matplotlib 의 "
    "savefig 로 저장하고, 저장한 뒤 os.path.getsize 로 크기를 print 해 0 이 아닌지 확인한다. "
    "그래프 코드는 맨 위에 다음 네 줄을 그대로 넣는다. 창을 띄우지 않고, 한글이 네모로 깨지지 않게 하려는 것이다.\n"
    "import matplotlib\n"
    "matplotlib.use('Agg')\n"
    "import matplotlib.pyplot as plt\n"
    f"plt.rcParams['font.family'] = '{FONT}'\n"
    "마크다운·텍스트만 files_write_file 로 저장한다. "
    "파일을 저장할 때는 폴더 경로를 적지 않는다. 파일 이름만 적는다. "
    "두 도구 모두 저장 폴더에서 실행되므로 이름만 적으면 그 폴더에 저장된다."
)

# 계획 장치(middleware) 없이 도구만 붙인 에이전트 - 비교용
# timeout 을 준다. 기본값은 요청 하나를 10분까지 기다리고 두 번 더 재시도해서,
# 응답이 늦거나 분당 한도에 걸리면 화면만 보고는 멈춘 것과 구별되지 않는다.
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=60)

# 모델 호출 횟수 상한. 반복에 빠져도 예외 없이 상한에서 스스로 끝난다.
LIMIT = ModelCallLimitMiddleware(run_limit=20, exit_behavior="end")
plain_agent = create_agent(model, TOOLS, system_prompt=SYSTEM_BASE, middleware=[LIMIT])
print("계획 없는 에이전트 준비 완료")

In [ ]:
# 한 문장에 네 가지 일(집계 2개·계산·그래프)을 담은 복합 요청
complex_q = ("chinook DB 를 분석해줘. 연도별 매출 합계와 국가별 매출 상위 5개를 구하고, "
             "연도별 매출의 전년 대비 증감률도 계산하고, 연도별 매출 막대그래프도 저장해줘.")

result_plain = await plain_agent.ainvoke({"messages": complex_q})
print_trajectory(result_plain)
print("\n불린 도구:", tool_names(result_plain))

> 계획 장치가 없으면 결과에 **`todos`(할 일 목록)가 없습니다**. 에이전트가 도구를 이어 부르긴 하지만,
> **무엇을 언제 할지 계획을 명시적으로 남기지 않습니다**. 단계가 더 많아지면 놓치는 일이 생깁니다.
> 실제로 위 기록에서 **그래프 저장이 빠졌는지** 확인해 보세요. 이제 **계획을 세우는 장치**를 끼워 봅니다.

---
# 2. 에이전트의 동작을 확장하는 미들웨어

## 미들웨어는 무엇인가

`create_agent` 가 만드는 에이전트는 **모델 호출 → 도구 실행 → 다시 모델 호출** 을 반복하는 루프입니다.
**미들웨어(middleware)** 는 이 루프의 코드를 고치지 않고, 루프의 정해진 지점에서 **내 코드가 대신 실행되게** 하는 장치입니다.

쓰는 법은 두 단계입니다.

1. `AgentMiddleware` 를 **상속**한 클래스를 만들고, **정해진 이름의 메서드**를 구현합니다.
2. 그 인스턴스를 `create_agent(..., middleware=[...])` 에 넘깁니다.

메서드 이름이 정해져 있는 이유는, 에이전트가 실행 중에 **그 이름으로 메서드를 찾아 호출**하기 때문입니다.
구현하지 않은 메서드는 기본 구현(아무것도 하지 않음)이 쓰이므로, **필요한 지점만 골라 구현**합니다.

## 훅: 미들웨어가 걸리는 자리

라이브러리가 루프에 **미리 열어 둔 자리**를 **훅(hook)** 이라고 합니다. 이름이 정해진 메서드 하나가 훅 하나입니다.
**훅은 자리, 미들웨어는 그 자리에 거는 물건**입니다. 미들웨어 하나가 훅을 여러 개 쓸 수도, 한 훅에 미들웨어가 여러 개 걸릴 수도 있습니다.
루프를 직접 고치면 버전이 오를 때 깨지고 여러 기능이 서로 충돌하니, 확장 지점을 이렇게 규격으로 정해 둔 것입니다.

그래서 미들웨어를 만들 때 첫 질문은 "무엇을 할까" 가 아니라 **"어느 훅에서 해야 할 수 있는 일인가"** 입니다.
자리를 잘못 고르면 그 자리에서는 그 일을 할 방법이 아예 없습니다.

훅 **자리는 여섯 개**가 전부입니다. 자리마다 이름이 둘씩 있는데(동기 `before_model`, 비동기 `abefore_model`) 하는 일은 같습니다. 아래 표는 동기용 이름으로 적었습니다.

> **표준이 아니라 LangChain 이 정한 방법입니다.** 정해진 지점에 남의 코드를 끼운다는 발상은 Django·Express 에도 있는 오래된 것이지만, **훅 여섯 자리의 이름과 인자는 LangChain 것**이라 다른 프레임워크로 옮기면 달라집니다. 누가 만든 서버든 붙는 공개 규격 **MCP 와는 다릅니다.**

## 구현할 수 있는 여섯 훅

<img src="../images/middleware_hooks.png" width="860">

| 훅(메서드) | 언제 호출되나 | 주로 이런 때 씁니다 |
|---|---|---|
| `before_agent` | 실행 시작 전 **1번** | 실행 전 준비. 지난 대화 기록을 불러와 상태에 넣기, 시작 시각·사용자 정보를 남기기 |
| `before_model` | 모델을 부르기 전 **매번** | 부르기 직전 손질. 지금까지의 호출 횟수를 세어 상한에 걸렸는지 보기(`ModelCallLimitMiddleware`), 길어진 대화의 앞부분을 요약해 상태에 넣기(`SummarizationMiddleware`) |
| `wrap_model_call` | 모델 호출을 **감싸서** | 호출을 내가 하기. 실패하면 한 번 더 부르기, 다른 모델로 바꿔 부르기(`ModelFallbackMiddleware`), 요청에 안내문을 덧붙여 부르기(`TodoListMiddleware`) |
| `after_model` | 모델이 답한 뒤 **매번** | 받은 답을 검사. 모델이 **고른 도구 호출을 사람이 승인**한 뒤 진행하게 하기(`HumanInTheLoopMiddleware`), 답변에 개인정보가 있는지 보고 가리기(`PIIMiddleware`), 형식이 어긋난 응답을 되돌려 다시 답하게 하기(`TodoListMiddleware`) |
| `wrap_tool_call` | 도구 호출을 **감싸서** | 도구 호출을 내가 하기. 실패한 도구를 다시 부르기(`ToolRetryMiddleware`), 너무 긴 도구 결과를 잘라 내기, 도구를 아예 부르지 않고 막기 |
| `after_agent` | 실행이 끝난 뒤 **1번** | 끝난 뒤 마무리. 최종 답을 파일·DB 에 남기기, 이번 실행의 도구 호출 수·토큰을 집계해 기록하기 |

여섯 개를 다 구현하지 않습니다. 대개 한두 개만 구현합니다.

> 괄호 안의 미들웨어 이름은 **아래 코드로 직접 확인한 것**입니다. `HumanInTheLoopMiddleware` 가 `wrap_tool_call` 이 아니라 `after_model` 인 것이 뜻밖일 수 있는데, **승인을 받는 대상이 도구 실행이 아니라 '모델이 그 도구를 고른 결정'** 이기 때문입니다. 모델이 답한 직후에 붙잡아야 도구가 실행되기 전에 멈출 수 있습니다.

## `before_`/`after_` 와 `wrap_` 은 할 수 있는 일이 다르다

인자를 보면 차이가 드러납니다.

- **`before_`/`after_`** 는 `state` 를 받습니다. 그 시점의 **상태를 읽고 바꾸는 것**까지 할 수 있습니다.
  호출 자체에는 손대지 못합니다.
- **`wrap_`** 은 `handler`, 곧 **호출 그 자체**를 인자로 받습니다. 그래서 `handler` 를 **여러 번 부르거나(재시도)
  한 번도 부르지 않을(차단)** 수 있고, 부르기 전에 요청을, 부른 뒤에 응답을 바꿀 수 있습니다.

재시도·대체 모델처럼 **호출을 다시 하거나 막아야 하는 기능**이 `wrap_` 자리에 있는 것은 이 때문입니다.

## 지점 말고도 붙는 것 두 가지

미들웨어는 실행 지점에 끼어드는 것 외에 두 가지를 더 할 수 있습니다.

- **도구를 추가한다**: `TodoListMiddleware` 는 `write_todos` 도구를 에이전트의 도구 목록에 넣습니다.
- **상태를 넓힌다**: 같은 미들웨어가 상태에 `todos` 키를 더해, 결과에서 `result["todos"]` 로 꺼낼 수 있게 합니다.

그래서 미들웨어 한 줄을 끼우면 동작만이 아니라 **모델이 쓸 수 있는 도구와 결과에 담기는 값**까지 함께 바뀝니다.

## 코드로 보면 이렇게 다릅니다

```python
def before_model(self, state, runtime):
    ...                            # 부르기 전에 할 일만 적는다

def wrap_model_call(self, request, handler):
    ...                            # 부르기 전
    response = handler(request)    # <- 호출이 내 손에 있다
    ...                            # 부른 뒤
    return response
```

> 그래서 **재시도·대체 모델은 `wrap_` 에서만** 만들 수 있습니다. 한 번 더 부르려면 호출을 쥐고 있어야 하니까요.

## 여러 개를 끼우면 순서는

`middleware=[A(), B()]` 로 넘깁니다. **먼저 적은 A 가 바깥, B 가 안쪽**인 양파 구조입니다.

- `before_` 는 **적은 순서**(A → B)
- `after_` 는 **거꾸로**(B → A)
- `wrap_` 은 **A 가 B 를 감싼다**

여섯 훅을 모두 구현한 `A`·`B` 를 끼우고 도구를 한 번 쓰는 질문을 던지면 이렇게 찍힙니다.

```text
A.before_agent              <- 루프 밖, 한 번
B.before_agent
   A.before_model           <- 1번째 모델 호출
   B.before_model
   A.wrap_model_call  시작
   B.wrap_model_call  시작
   B.wrap_model_call  끝
   A.wrap_model_call  끝
   B.after_model            <- after 는 역순
   A.after_model
   A.wrap_tool_call   시작    <- 모델이 도구를 부르겠다고 해서 열렸다
   B.wrap_tool_call   시작
   B.wrap_tool_call   끝
   A.wrap_tool_call   끝
   A.before_model           <- 도구 결과를 들고 2번째 모델 호출
   ...(같은 순서로 반복)
B.after_agent               <- 루프 밖, 한 번
A.after_agent
```

## 어떤 미들웨어가 있나

| 갈래 | 미들웨어 | 하는 일 | 언제 |
|---|---|---|---|
| **계획** | `TodoListMiddleware` | 복합 요청에 **할 일 목록**을 세우고 추적 | **오늘** |
| **대화 관리** | `SummarizationMiddleware` | 길어진 대화의 **앞부분을 요약** | 메모리·컨텍스트 단원 |
| | `ContextEditingMiddleware` | 오래된 **도구 결과를 잘라** 냄 | 메모리·컨텍스트 단원 |
| **안전** | `HumanInTheLoopMiddleware` | 위험한 도구 앞에서 **사람의 승인** | 사이클·HITL 단원 |
| | `PIIMiddleware` | 개인정보를 **가리거나 막음** | 메모리·컨텍스트 단원 |
| **한도** | `ModelCallLimitMiddleware` | 모델 **호출 횟수 상한** | **오늘** |
| | `ToolCallLimitMiddleware` | 도구 **호출 횟수 상한** | 참고 |
| **견고성** | `ModelRetryMiddleware` · `ToolRetryMiddleware` | 실패하면 **다시 시도** | 참고 |
| | `ModelFallbackMiddleware` | 실패하면 **다른 모델로** | 참고 |

> **고르는 기준**: 판단이 문제면 계획, 대화가 길면 대화 관리, 위험하면 안전, 폭주·요금이면 한도, 간헐적 실패면 견고성.
> 없으면 직접 만들 수도 있지만, 이 과정에서는 **골라 끼우는** 데까지 합니다.

> **한도 미들웨어는 이 강의에서 이미 쓰고 있습니다.** 1절과 앞에서 만든 에이전트에
> `ModelCallLimitMiddleware(run_limit=20, exit_behavior="end")` 를 끼웠습니다.
> 모델이 같은 도구 호출을 반복해도 20번에서 스스로 멈추고, `exit_behavior="end"` 라서
> 예외가 아니라 **그때까지의 기록이 그대로** 돌아옵니다. 상한이 없으면 반복이 끝나지 않아
> 화면이 멈춘 것처럼 보이고 요금만 쌓입니다.

> MCP 서버를 쓸 때 특히 쓸모 있는 것이 **`HumanInTheLoopMiddleware`** 입니다.
> 남의 서버가 준 도구에는 `write_query`·`write_file` 처럼 **바꾸는 도구**가 섞여 있습니다.
> 앞 단원에서는 그런 도구를 **아예 넘기지 않는** 방법을 썼고, 꼭 필요하면 이 미들웨어로 **사람의 승인**을 받게 합니다.

표가 맞는지 확인해 봅니다. 기본 클래스(`AgentMiddleware`)와 **달라진 훅**이 그 미들웨어가 쓰는 자리입니다.

In [ ]:
from langchain.agents.middleware import (AgentMiddleware, HumanInTheLoopMiddleware,
                                         SummarizationMiddleware, TodoListMiddleware,
                                         ToolCallLimitMiddleware, ToolRetryMiddleware)

HOOKS = ['before_agent', 'before_model', 'wrap_model_call',
         'after_model', 'wrap_tool_call', 'after_agent']


def used_hooks(mw_class):
    """그 미들웨어가 실제로 구현한 훅 이름만 골라 돌려준다(동기 이름 기준)."""
    # 기본 클래스의 훅과 '다른 함수' 로 바뀌어 있으면 그 자리를 쓴 것이다.
    # 훅 이름이 문자열 목록으로 오므로 mw_class.h 처럼 점으로는 못 꺼낸다. 그래서 이름으로 조회한다.
    return [h for h in HOOKS
            if getattr(mw_class, h) is not getattr(AgentMiddleware, h)]


for mw in (TodoListMiddleware, SummarizationMiddleware, HumanInTheLoopMiddleware,
           ToolCallLimitMiddleware, ToolRetryMiddleware):
    print(f'{mw.__name__:28} {used_hooks(mw)}')

하나같이 한두 자리만 씁니다. 그 자리가 왜 거기인지도 읽힙니다.

- **`SummarizationMiddleware`** → `before_model`: 대화를 줄이려면 **부르기 전**이어야 합니다.
- **`HumanInTheLoopMiddleware`** → `after_model`: 모델이 **도구를 부르겠다고 말한 직후**가 물어볼 자리입니다.
- **`ToolRetryMiddleware`** → `wrap_tool_call`: 다시 부르려면 호출을 쥐어야 합니다.
- **`TodoListMiddleware`**(오늘) → `wrap_model_call` 과 `after_model` 두 자리: 계획을 세우라는 안내문을 요청에 덧붙이고, 한 응답에 계획이 두 번 들어오는 것을 막습니다.

> 미들웨어는 **도구를 함께 들고 오기도** 합니다. 계획을 적는 `write_todos` 가 바로 `TodoListMiddleware` 가 추가하는 도구입니다.
> 끼우는 것만으로 도구가 하나 늘어납니다. **MCP 서버가 도구를 주는 것과 같은 자리**에 도구가 하나 더 붙는 셈입니다.

---
# 3. 계획을 세우는 `TodoListMiddleware`

## 이 방식의 이름은 Plan-and-Execute

**먼저 계획을 세우고(Plan), 그 계획을 한 단계씩 실행(Execute)** 하는 방식입니다.

앞서 배운 **ReAct** 와 견주면 차이가 하나입니다.

| | 전체 그림 | 단계가 많아지면 |
|---|---|---|
| **ReAct** | 그리지 않는다. "생각 → 행동 → 관찰" 을 한 걸음씩 반복할 뿐 | 무엇을 남겨 뒀는지 놓친다 |
| **Plan-and-Execute** | **할 일 목록을 먼저 문서로 남긴다** | 목록을 보며 움직이므로 진행 상황을 눈으로 추적할 수 있다 |

## 쓰는 법

LangChain 에서는 직접 구현하지 않고 **`TodoListMiddleware`** 를 끼워 얻습니다.
붙인 도구는 그대로 두고 두 가지만 바꿉니다.

1. `create_agent` 에 **`middleware=[TodoListMiddleware()]`** 를 넘긴다
2. 시스템 프롬프트에 **"복합 요청은 먼저 계획을 세워라"** 를 적는다

<img src="../images/react_vs_plan_execute.png" width="760">

*왼쪽은 앞서 배운 ReAct, 오른쪽이 오늘 만들 방식입니다. 차이는 **할 일 목록을 남기느냐** 하나입니다.*

In [ ]:
# 도구는 앞에서 고른 그대로다. 달라지는 것은 middleware 한 줄과 프롬프트 한 문장이다.
SYSTEM_PLAN = SYSTEM_BASE + (
    " 여러 단계가 필요한 복합 요청은 먼저 계획을 세우고 단계별로 처리하라."
)

analyst = create_agent(model, TOOLS,
                       middleware=[TodoListMiddleware(), LIMIT],
                       system_prompt=SYSTEM_PLAN)
print("계획을 세우는 분석 에이전트 준비 완료")

이제 같은 복합 요청을 **계획을 세우는 에이전트**에 던지고, 메시지 기록과 **할 일 목록**을 관찰합니다.

> 아래 질문 끝에 "먼저 계획을 세우고 진행해줘" 를 한 번 더 적었습니다. 시스템 프롬프트만으로도 대개 계획을 세우지만, 수업에서 **한 번에 보이게** 하려고 못 박은 것입니다. 계획을 세우게 만드는 것은 어디까지나 **미들웨어와 시스템 프롬프트**이고, 이 문장은 거들 뿐입니다.

In [ ]:
# 같은 모양의 복합 요청을 계획 에이전트(analyst)에 던진다.
# 볼 것은 하나다. 계획을 먼저 세우는가, 그리고 네 가지를 다 해내는가.
plan_q = ("chinook DB 에서 아티스트별 앨범 수와 앨범별 곡 수를 구하고, "
          "앨범이 가장 많은 아티스트 3팀이 전체 앨범에서 차지하는 비중(%)도 계산한 다음, "
          "아티스트별 앨범 수 상위 10팀 막대그래프를 아티스트별_앨범수.png 라는 이름으로 저장해줘. "
          "먼저 어떻게 처리할지 계획을 세우고 진행해줘.")

result = await analyst.ainvoke({"messages": plan_q})
print_trajectory(result)

print("\n---- 세운 계획과 진행 상태 ----")
print("계획(todos)이 있나?:", "todos" in result)
# 계획을 세우지 않은 실행도 있을 수 있으므로 get 으로 안전하게 꺼낸다
for item in result.get("todos", []):
    print(f"- [{item['status']}] {item['content']}")

> 방금 출력된 메시지 기록에서 **`write_todos`** 를 찾아보세요. 대개 **맨 처음** 이 도구로 **할 일 목록**을 세우고,
> 각 단계를 처리한 뒤 다시 `write_todos` 로 상태를 갱신합니다.
> 계획을 **눈에 보이게** 만들면, 복합 요청도 단계를 빠뜨리지 않고 처리할 수 있습니다.

> 계획을 **몇 개로 쪼갤지, 어떤 문장으로 적을지는 모델이 정합니다**. 실제 모델을 부르므로 실행할 때마다 항목 수와 문구가 달라질 수 있습니다.
> 우리가 확인할 것은 "계획이 세워졌고 단계별로 처리됐는가"이지, 항목이 정확히 몇 개인지가 아닙니다.

---
## 이번 강의 정리

| 개념 | 하는 일 |
|---|---|
| 복합 요청의 한계 | 계획 없이 여러 단계를 처리하면 단계를 놓치기 쉬움 |
| 미들웨어 | 에이전트 루프 주변에 기능을 덧붙이는 확장 장치. 정해진 **훅** 자리에만 건다(자리를 정한 것은 LangChain) |
| 훅 여섯 자리 | `before_agent` · `before_model` · `wrap_model_call` · `after_model` · `wrap_tool_call` · `after_agent` |
| `before`·`after` 와 `wrap` | 앞의 둘은 **상태**를 읽고 바꾸는 자리, `wrap` 은 **호출 자체**를 쥐는 자리(재시도·차단은 `wrap` 에서만) |
| `TodoListMiddleware` | 복합 요청에 **할 일 목록(계획)** 을 세우고 상태를 추적. `write_todos` 도구를 함께 들고 온다 |
| **Plan-and-Execute** | 계획을 먼저 세우고(Plan) 단계별로 실행(Execute): 위 미들웨어로 얻는 방식의 이름 |

- 도구가 **손**이라면, 계획(todos)은 에이전트의 **작업 순서표**입니다.
- `result.get('todos', [])` 로 계획과 진행 상태를 **눈으로 확인**할 수 있습니다.
- **도구가 남의 것이어도 계획을 세우는 자리는 그대로**입니다. 미들웨어는 도구가 아니라 **루프**에 걸리기 때문입니다.
- 미들웨어를 만들 때 첫 질문은 "무엇을 할까" 가 아니라 **"어느 훅에서 해야 할 수 있는 일인가"** 입니다.